## Stage 2 - Data Acquisition, Exploration, and Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv('dontpatronizeme_pcl.tsv', sep='\t', header=None,
                 names=['par_id','art_id','keyword','country','text','label'])

train_ids = pd.read_csv('train_semeval_parids-labels.csv')
dev_ids   = pd.read_csv('dev_semeval_parids-labels.csv')

train = df[df['par_id'].isin(train_ids['par_id'])].copy()
dev   = df[df['par_id'].isin(dev_ids['par_id'])].copy()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Class distribution
print(train['label'].value_counts())
print(train['label'].value_counts(normalize=True))  # imbalance ratio

# Sequence length
train['token_count'] = train['text'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train['label'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','tomato'])
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['No PCL', 'PCL'], rotation=0)

train.groupby('label')['token_count'].plot(kind='kde', ax=axes[1], legend=True)
axes[1].set_title('Sequence Length Distribution by Class')
axes[1].set_xlabel('Token Count')

plt.tight_layout()
plt.savefig('figures/class_dist_seqlen.pdf', bbox_inches='tight')

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

def top_ngrams(corpus, n=2, top_k=15):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(corpus)
    bag = vec.transform(corpus)
    freqs = zip(vec.get_feature_names_out(), np.asarray(bag.sum(axis=0)).ravel())
    return sorted(freqs, key=lambda x: x[1], reverse=True)[:top_k]

pcl_text    = train[train['label']==1]['text']
no_pcl_text = train[train['label']==0]['text']

pcl_bigrams    = top_ngrams(pcl_text,    n=2)
no_pcl_bigrams = top_ngrams(no_pcl_text, n=2)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title in zip(axes,
                            [pcl_bigrams, no_pcl_bigrams],
                            ['Top Bigrams — PCL', 'Top Bigrams — No PCL']):
    labels, counts = zip(*data)
    ax.barh(labels, counts)
    ax.set_title(title)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('figures/ngram_divergence.pdf', bbox_inches='tight')

In [ ]:
topic_pcl = train.groupby('keyword')['label'].mean().sort_values(ascending=False)
topic_pcl.plot(kind='bar', figsize=(14, 4), title='PCL Rate by Topic Keyword')
plt.ylabel('Proportion Labelled PCL')
plt.tight_layout()
plt.savefig('figures/pcl_by_topic.pdf', bbox_inches='tight')